# SNS-Assistant — 1단계: 개발 환경 구성 & Figma 연동 검증

| 셀 | 내용 |
|---|---|
| **Cell 1** | 필수 패키지 설치 및 설치 확인 |
| **Cell 2** | `.env` 템플릿 생성 → 환경 변수 로드/검증 (`.gitignore` 자동 등록) |
| **Cell 3** | Figma REST API 토큰 검증 (`/v1/me`, `/v1/files/{key}`) + 로컬 Figma MCP 서버 확인(선택) |

> **VS Code 커널 선택**: 우측 상단 *Select Kernel* → 프로젝트 가상환경(`.venv`)을 선택한 뒤 실행하세요.
> 가상환경이 없다면 터미널(PowerShell)에서: `python -m venv .venv` → `.venv\Scripts\Activate.ps1` → `pip install ipykernel`

In [1]:
# ============================================================
# Cell 1. 필수 라이브러리 설치 및 확인
# ============================================================
# %pip 는 "현재 실행 중인 커널"의 파이썬에 설치하므로 가상환경에서도 안전합니다.
%pip install -q --upgrade python-dotenv google-genai requests boto3

import sys
from importlib.metadata import version, PackageNotFoundError

REQUIRED = {
    "python-dotenv": "dotenv",
    "google-genai":  "google.genai",
    "requests":      "requests",
    "boto3":         "boto3",
}

print(f"🐍 Python : {sys.version.split()[0]}")
print(f"📂 실행 경로: {sys.executable}")
in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
print(f"{'✅' if in_venv else '⚠️ '} 가상환경 {'사용 중' if in_venv else '미사용 (전역 Python에 설치됨)'}\n")

failed = []
for pkg, module in REQUIRED.items():
    try:
        __import__(module)
        print(f"  ✅ {pkg:<15} v{version(pkg)}")
    except (ImportError, PackageNotFoundError) as e:
        failed.append(pkg)
        print(f"  ❌ {pkg:<15} 설치/임포트 실패 → {e}")

print("\n" + ("🎉 [성공] 모든 패키지 준비 완료" if not failed
              else f"🚨 [실패] {failed} — 커널 재시작(Restart) 후 다시 실행해 보세요."))

Note: you may need to restart the kernel to use updated packages.
🐍 Python : 3.14.7
📂 실행 경로: c:\dev\claude-code-agent-course\chapter11\ax-job-agent\.venv\Scripts\python.exe
✅ 가상환경 사용 중

  ✅ python-dotenv   v1.2.3
  ✅ google-genai    v2.25.0
  ✅ requests        v2.34.2
  ✅ boto3           v1.43.102

🎉 [성공] 모든 패키지 준비 완료


In [ ]:
# ============================================================
# Cell 2. .env 파일 생성 / 로드 / 검증
# ============================================================
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_PATH       = Path.cwd() / ".env"
GITIGNORE_PATH = Path.cwd() / ".gitignore"

ENV_TEMPLATE = """# ------------------------------------------------------------
# SNS-Assistant 환경 변수 (이 파일은 절대 Git에 커밋하지 마세요!)
# ------------------------------------------------------------

# [Figma] Personal Access Token
#   발급: Figma → 좌측 상단 계정 메뉴 → Settings → Security → Personal access tokens
#   권한(scope): 최소 'current_user:read', 'file_content:read' 체크
FIGMA_ACCESS_TOKEN=your_figma_personal_access_token

# [Figma] 테스트용 File Key (또는 파일 URL 전체를 붙여넣어도 됩니다)
#   예) https://www.figma.com/design/AbCdEf123456/My-File  →  AbCdEf123456
FIGMA_FILE_KEY=your_figma_file_key

# [Gemini] API Key — 발급: https://aistudio.google.com/apikey
GEMINI_API_KEY=your_gemini_api_key

# [AWS] IAM 액세스 키 (S3 업로드 등에 사용 예정)
AWS_ACCESS_KEY_ID=your_aws_access_key_id
AWS_SECRET_ACCESS_KEY=your_aws_secret_access_key
AWS_DEFAULT_REGION=ap-northeast-2
AWS_S3_BUCKET=your_s3_bucket_name
"""

# 1) .env 파일이 없으면 템플릿 생성 (기존 파일은 절대 덮어쓰지 않음)
if ENV_PATH.exists():
    print(f"ℹ️  기존 .env 파일 사용: {ENV_PATH}")
else:
    ENV_PATH.write_text(ENV_TEMPLATE, encoding="utf-8")
    print(f"🆕 .env 템플릿 생성 완료: {ENV_PATH}")
    print("   👉 VS Code에서 .env 를 열어 실제 값으로 교체한 뒤 이 셀을 다시 실행하세요.")

# 2) .gitignore 에 .env 등록 (키 유출 방지)
ignore_lines = GITIGNORE_PATH.read_text(encoding="utf-8").splitlines() if GITIGNORE_PATH.exists() else []
if ".env" not in ignore_lines:
    with GITIGNORE_PATH.open("a", encoding="utf-8") as f:
        f.write(("\n" if ignore_lines and ignore_lines[-1] else "") + ".env\n.venv/\n.ipynb_checkpoints/\n")
    print("🔒 .gitignore 에 .env 추가 완료")
else:
    print("🔒 .gitignore 에 .env 이미 등록됨")

# 3) 환경 변수 로드 (override=True → .env 수정 후 셀만 재실행해도 반영)
load_dotenv(ENV_PATH, override=True, encoding="utf-8")

def mask(value: str) -> str:
    return value[:4] + "*" * 8 + value[-4:] if len(value) > 12 else "*" * len(value)

REQUIRED_KEYS = ["FIGMA_ACCESS_TOKEN", "FIGMA_FILE_KEY", "GEMINI_API_KEY",
                 "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_DEFAULT_REGION", "AWS_S3_BUCKET"]

print("\n📋 환경 변수 점검")
missing = []
for key in REQUIRED_KEYS:
    val = (os.getenv(key) or "").strip()
    if not val or val.startswith("your_"):
        missing.append(key)
        print(f"  ❌ {key:<22} 미설정")
    else:
        shown = val if key in ("AWS_DEFAULT_REGION", "AWS_S3_BUCKET") else mask(val)
        print(f"  ✅ {key:<22} {shown}")

print("\n" + ("🎉 [성공] 모든 환경 변수 로드 완료" if not missing
              else f"⚠️  [미완료] {len(missing)}개 미설정 — Figma 검증에는 FIGMA_ACCESS_TOKEN 만 필수입니다."))

In [ ]:
# ============================================================
# Cell 3. Figma Access Token 검증 (REST API) + 로컬 Figma MCP 서버 확인
# ============================================================
import os, re, requests

FIGMA_API = "https://api.figma.com/v1"
TOKEN     = (os.getenv("FIGMA_ACCESS_TOKEN") or "").strip()
RAW_KEY   = (os.getenv("FIGMA_FILE_KEY") or "").strip()

def extract_file_key(raw: str) -> str:
    """파일 URL 전체를 넣어도 File Key 만 추출 (design / file / proto / board 지원)."""
    m = re.search(r"figma\.com/(?:design|file|proto|board)/([A-Za-z0-9]+)", raw)
    return m.group(1) if m else raw

def figma_get(path: str, **params):
    return requests.get(f"{FIGMA_API}{path}", headers={"X-Figma-Token": TOKEN},
                        params=params, timeout=15)

ERROR_HINT = {
    400: "요청 형식 오류 (File Key 형식을 확인하세요)",
    403: "토큰이 유효하지 않거나 만료됨 / scope 부족",
    404: "파일을 찾을 수 없음 (File Key 오타 또는 접근 권한 없음)",
    429: "요청 한도 초과 (Rate limit) — 잠시 후 재시도",
}

results = {}

# ---------- [1] 토큰 → 사용자 정보 ----------
print("🔑 [1/3] Figma 토큰 검증 (GET /v1/me)")
if not TOKEN or TOKEN.startswith("your_"):
    print("  ❌ FIGMA_ACCESS_TOKEN 미설정 → Cell 2 에서 .env 를 먼저 채워주세요.")
    results["token"] = False
else:
    try:
        r = figma_get("/me")
        if r.ok:
            me = r.json()
            print(f"  ✅ 인증 성공 → {me.get('handle')} ({me.get('email')})  id={me.get('id')}")
            results["token"] = True
        else:
            print(f"  ❌ 인증 실패 [HTTP {r.status_code}] {ERROR_HINT.get(r.status_code, '')}")
            print(f"     응답: {r.text[:200]}")
            results["token"] = False
    except requests.RequestException as e:
        print(f"  ❌ 네트워크 오류: {e}")
        results["token"] = False

# ---------- [2] File Key → 파일 정보 ----------
print("\n📄 [2/3] 테스트 파일 조회 (GET /v1/files/{key}?depth=1)")
FILE_KEY = extract_file_key(RAW_KEY)
if not results["token"]:
    print("  ⏭️  토큰 검증 실패로 건너뜀")
    results["file"] = False
elif not FILE_KEY or FILE_KEY.startswith("your_"):
    print("  ⏭️  FIGMA_FILE_KEY 미설정 → 건너뜀 (선택 항목)")
    results["file"] = None
else:
    try:
        r = figma_get(f"/files/{FILE_KEY}", depth=1)   # depth=1: 페이지 목록만 받아 빠르게 응답
        if r.ok:
            data  = r.json()
            pages = data.get("document", {}).get("children", [])
            print(f"  ✅ 파일명      : {data.get('name')}")
            print(f"     최종 수정   : {data.get('lastModified')}")
            print(f"     버전        : {data.get('version')}")
            print(f"     페이지 {len(pages)}개 : {[p.get('name') for p in pages]}")
            results["file"] = True
        else:
            print(f"  ❌ 조회 실패 [HTTP {r.status_code}] {ERROR_HINT.get(r.status_code, '')}")
            print(f"     File Key: {FILE_KEY} / 응답: {r.text[:200]}")
            results["file"] = False
    except requests.RequestException as e:
        print(f"  ❌ 네트워크 오류: {e}")
        results["file"] = False

# ---------- [3] 로컬 Figma MCP 서버 (선택) ----------
# Figma 데스크톱 앱 → Preferences → 'Enable Dev Mode MCP Server' 활성화 시 로컬에 열리는 서버
print("\n🧩 [3/3] 로컬 Figma MCP 서버 확인 (http://127.0.0.1:3845/mcp) — 선택")
try:
    requests.get("http://127.0.0.1:3845/mcp", timeout=2)   # 응답 코드와 무관하게 포트가 열려 있으면 실행 중
    print("  ✅ Figma MCP 서버 실행 중 (VS Code/Claude 등 MCP 클라이언트에서 연결 가능)")
    results["mcp"] = True
except requests.RequestException:
    print("  ⏭️  MCP 서버 미실행 — REST API 방식만으로도 이후 단계 진행 가능합니다.")
    results["mcp"] = None

# ---------- 최종 요약 ----------
label = {True: "✅ 성공", False: "❌ 실패", None: "⏭️  건너뜀"}
print("\n" + "=" * 50)
print(f"  Figma 토큰 인증 : {label[results['token']]}")
print(f"  Figma 파일 조회 : {label[results['file']]}")
print(f"  로컬 MCP 서버   : {label[results['mcp']]}")
print("=" * 50)
print("🎉 [성공] Figma 연동 준비 완료 — 2단계로 진행하세요." if results["token"] and results["file"] is not False
      else "🚨 [실패] 위 오류 메시지를 확인 후 .env 를 수정하고 Cell 2 → Cell 3 순서로 재실행하세요.")

In [ ]:
# ============================================================
# Cell 4. Gemini 멀티모달 호출 → SNS 게시글(본문 + 해시태그) 생성 검증
# ============================================================
import os, time, textwrap
from pathlib import Path

import requests
from dotenv import load_dotenv
from google import genai
from google.genai import types, errors as genai_errors
from pydantic import BaseModel, Field

load_dotenv(Path.cwd() / ".env", override=True, encoding="utf-8")   # Cell 2 없이 단독 실행해도 동작

MODEL        = "gemini-3.8-flash"   # gemini-2.5-flash 는 신규 키에서 404 (지원 종료)
KEYWORDS     = ["아메리카노", "감성 카페", "주말 여유"]
MAX_RETRIES  = 4          # 최대 시도 횟수 (429 / 5xx 일시 오류 시 재시도)
SAMPLE_DIR   = Path.cwd() / "samples"
SAMPLE_IMAGE = SAMPLE_DIR / "sample_coffee.jpg"
SAMPLE_URL   = "https://upload.wikimedia.org/wikipedia/commons/4/45/A_small_cup_of_coffee.JPG"
MIME_TYPES   = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png", ".webp": "image/webp"}

PROMPT = "이미지와 주어진 키워드를 바탕으로 인스타그램 스타일의 매력적인 SNS 게시글 본문과 해시태그 5개 이상을 작성해줘."

class SnsPost(BaseModel):
    """Gemini 응답 스키마 — JSON으로 받아 바로 파싱한다."""
    body: str = Field(description="인스타그램 스타일 게시글 본문 (해시태그 제외)")
    hashtags: list[str] = Field(description="'#'으로 시작하는 해시태그 5개 이상")

results = {}

# ---------- [1] Gemini Client 초기화 ----------
print("🤖 [1/4] Gemini Client 초기화")
API_KEY = (os.getenv("GEMINI_API_KEY") or "").strip()
client = None
if not API_KEY or API_KEY.startswith("your_"):
    print("  ❌ GEMINI_API_KEY 미설정 → .env 에 키를 입력한 뒤 다시 실행하세요.")
else:
    try:
        client = genai.Client(api_key=API_KEY)
        print(f"  ✅ Client 생성 완료 (모델: {MODEL})")
    except Exception as e:
        print(f"  ❌ Client 생성 실패: {e}")
results["client"] = client is not None

# ---------- [2] 샘플 이미지 준비 ----------
print("\n🖼️  [2/4] 샘플 이미지 준비")
image_bytes = None
if SAMPLE_IMAGE.exists():
    image_bytes = SAMPLE_IMAGE.read_bytes()
    print(f"  ✅ 로컬 이미지 사용: {SAMPLE_IMAGE} ({len(image_bytes) / 1024:.0f} KB)")
else:
    try:
        # Wikimedia 는 User-Agent 가 없으면 요청을 거부한다
        r = requests.get(SAMPLE_URL, headers={"User-Agent": "SNS-Assistant/0.1 (test)"}, timeout=30)
        r.raise_for_status()
        if not r.headers.get("Content-Type", "").startswith("image/"):
            raise ValueError(f"이미지가 아닌 응답: {r.headers.get('Content-Type')}")
        SAMPLE_DIR.mkdir(exist_ok=True)
        SAMPLE_IMAGE.write_bytes(r.content)
        image_bytes = r.content
        print(f"  🆕 공개 테스트 이미지 다운로드 완료: {SAMPLE_IMAGE} ({len(image_bytes) / 1024:.0f} KB)")
    except Exception as e:
        print(f"  ❌ 다운로드 실패: {e}")
        print(f"     👉 직접 준비하려면 이미지를 {SAMPLE_IMAGE} 에 저장하세요.")
results["image"] = image_bytes is not None

# ---------- [3] Gemini 멀티모달 호출 ----------
print("\n📡 [3/4] Gemini 멀티모달 호출 (이미지 + 키워드)")
print(f"  키워드: {', '.join(KEYWORDS)}")
post = None
if not (results["client"] and results["image"]):
    print("  ⏭️  이전 단계 실패로 건너뜀")
else:
    try:
        start = time.perf_counter()
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response = client.models.generate_content(
                    model=MODEL,
                    contents=[
                        types.Part.from_bytes(data=image_bytes,
                                              mime_type=MIME_TYPES.get(SAMPLE_IMAGE.suffix.lower(), "image/jpeg")),
                        f"{PROMPT}\n\n키워드: {', '.join(KEYWORDS)}",
                    ],
                    config=types.GenerateContentConfig(
                        response_mime_type="application/json",
                        response_schema=SnsPost,
                        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
                    ),
                )
                break
            except genai_errors.APIError as e:
                # 429(한도 초과) / 5xx(서버 과부하)는 일시적이므로 대기 후 재시도, 그 외 오류는 즉시 실패
                if e.code not in (429, 500, 503) or attempt == MAX_RETRIES:
                    raise
                wait = 2 ** attempt
                print(f"  ⏳ HTTP {e.code} — {wait}초 후 재시도 ({attempt}/{MAX_RETRIES - 1})")
                time.sleep(wait)
        elapsed = time.perf_counter() - start
        usage = response.usage_metadata
        print(f"  ✅ 응답 수신 ({elapsed:.1f}초, 입력 {usage.prompt_token_count} / 출력 {usage.candidates_token_count} 토큰)")

        # ---------- [4] 응답 파싱 ----------
        print("\n🧩 [4/4] 응답 파싱")
        post = response.parsed if isinstance(response.parsed, SnsPost) else SnsPost.model_validate_json(response.text)
        post.hashtags = [t if t.startswith("#") else f"#{t}" for t in (h.strip().replace(" ", "") for h in post.hashtags) if t]
        print(f"  ✅ 파싱 성공 — 본문 {len(post.body)}자, 해시태그 {len(post.hashtags)}개")
        if len(post.hashtags) < 5:
            print("  ⚠️  해시태그가 5개 미만입니다 (프롬프트 조건 미충족)")
    except Exception as e:
        print(f"  ❌ 호출/파싱 실패: {type(e).__name__}: {e}")
results["post"] = post is not None

# ---------- 결과 출력 ----------
if post:
    print("\n" + "─" * 50)
    print("📝 생성된 SNS 게시글")
    print("─" * 50)
    for para in post.body.splitlines():
        print(textwrap.fill(para, width=48) if para.strip() else "")
    print()
    print(" ".join(post.hashtags))
    print("─" * 50)

label = {True: "✅ 성공", False: "❌ 실패"}
print("\n" + "=" * 50)
print(f"  Gemini Client  : {label[results['client']]}")
print(f"  샘플 이미지    : {label[results['image']]}")
print(f"  게시글 생성    : {label[results['post']]}")
print("=" * 50)
print("🎉 [성공] Gemini 게시글 자동 작성 검증 완료" if all(results.values())
      else "🚨 [실패] 위 오류 메시지를 확인 후 다시 실행하세요.")

In [ ]:
# ============================================================
# Cell 5. Streamlit UI 프로토타입 (다중 이미지 업로드 + 비율별 센터 크롭/다운로드 + Mock 게시글)
# ============================================================
%pip install -q --upgrade streamlit pillow

import sys, time, socket, subprocess, textwrap
from pathlib import Path
from importlib.metadata import version

import requests

APP_PATH    = Path.cwd() / "app_prototype.py"
CONFIG_PATH = Path.cwd() / ".streamlit" / "config.toml"
LOG_PATH    = Path.cwd() / "streamlit.log"
HOST        = "0.0.0.0"   # 모든 네트워크 인터페이스에서 접속 허용 (같은 네트워크의 다른 기기)
PORT        = 8501
URL         = f"http://localhost:{PORT}"

# ---------- [1] 패키지 확인 ----------
print("📦 [1/4] 패키지 확인")
for pkg in ("streamlit", "pillow"):
    print(f"  ✅ {pkg:<10} v{version(pkg)}")

# ---------- [2] 테마 설정 (docs/design-system.md: Inter 폰트, 배경/본문 색상) ----------
CONFIG_TOML = '''\
# Cell 5 에서 자동 생성 — docs/design-system.md 기준
[theme]
font = "Inter:https://fonts.googleapis.com/css2?family=Inter:ital,wght@0,400;0,500;0,600;0,700;1,400&display=swap"
baseFontSize = 14

[theme.light]
backgroundColor = "#FFFFFF"
textColor = "#6B7280"

[theme.dark]
backgroundColor = "#111827"
textColor = "#6B7280"
'''

# ---------- [3] app_prototype.py ----------
APP_CODE = r'''
"""SNS Assistant UI 프로토타입 — SNS-Assistant.ipynb Cell 5 에서 자동 생성된다 (직접 수정 시 덮어써짐)."""
import html
import io
from pathlib import Path

import streamlit as st
from PIL import Image, ImageOps, UnidentifiedImageError

st.set_page_config(page_title="SNS Assistant", page_icon="📸", layout="wide")

# ---------- 디자인 토큰 (docs/design-system.md) ----------
PALETTE = {  # 텍스트 색상: 강조 / 보조 / 본문
    "light": {"strong": "#374151", "muted": "#9CA3AF", "body": "#6B7280"},
    "dark":  {"strong": "#E5E7EB", "muted": "#4B5563", "body": "#6B7280"},
}
c = PALETTE.get(st.context.theme.type or "light", PALETTE["light"])

st.markdown(f"""
<style>
.ds-metric   {{ font-size: 30px; font-weight: 600; line-height: normal; color: {c["strong"]}; margin: 0; }}
.ds-title    {{ font-size: 18px; font-weight: 500; line-height: 28px;   color: {c["strong"]}; margin: 0; }}
.ds-subtitle {{ font-size: 16px; font-weight: 500; line-height: normal; color: {c["muted"]};  margin: 0; }}
.ds-text     {{ font-size: 14px; font-weight: 400; line-height: 20px;   color: {c["body"]};   margin: 0; white-space: pre-wrap; }}
.ds-bold     {{ font-size: 14px; font-weight: 700; line-height: normal; color: {c["body"]}; }}
.ds-label    {{ font-size: 12px; font-weight: 400; line-height: normal; color: {c["body"]};   margin: 0; }}
</style>
""", unsafe_allow_html=True)


def ds(text: str, style: str) -> None:
    """디자인 시스템 텍스트 스타일(metric/title/subtitle/text/bold/label)로 출력."""
    st.markdown(f'<p class="ds-{style}">{html.escape(text)}</p>', unsafe_allow_html=True)


# ---------- 이미지 처리 ----------
RATIOS = {  # 라벨: (가로, 세로) — None 이면 원본 유지
    "1:1 (인스타그램 피드 / 정방형)": (1, 1),
    "9:16 (인스타 리얼스 / 틱톡 / 유튜브 쇼츠)": (9, 16),
    "4:5 (인스타그램 세로 피드)": (4, 5),
    "16:9 (가로형 / X / 페이스북)": (16, 9),
    "원본 비율 유지": None,
}
FORMATS = {  # 확장자: (Pillow 포맷, MIME)
    "jpg": ("JPEG", "image/jpeg"), "jpeg": ("JPEG", "image/jpeg"),
    "png": ("PNG", "image/png"),   "webp": ("WEBP", "image/webp"),
}


def center_crop(img: Image.Image, ratio: tuple[int, int] | None) -> Image.Image:
    """목표 비율에서 가능한 가장 큰 영역을 이미지 중앙에서 잘라낸다."""
    if ratio is None:
        return img
    w, h = img.size
    rw, rh = ratio
    if w * rh > h * rw:                      # 원본이 더 넓음 → 좌우를 자름
        new_w = round(h * rw / rh)
        left = (w - new_w) // 2
        return img.crop((left, 0, left + new_w, h))
    new_h = round(w * rh / rw)               # 원본이 더 높음 → 위아래를 자름
    top = (h - new_h) // 2
    return img.crop((0, top, w, top + new_h))


def encode(img: Image.Image, ext: str) -> bytes:
    fmt, _ = FORMATS[ext]
    if fmt == "JPEG" and img.mode != "RGB":  # JPEG 는 투명도(RGBA/P)를 저장할 수 없음
        img = img.convert("RGB")
    buf = io.BytesIO()
    img.save(buf, format=fmt, **({"quality": 95} if fmt in ("JPEG", "WEBP") else {}))
    return buf.getvalue()


def mock_post(keywords: list[str]) -> dict:
    """Gemini 연동 전 레이아웃 확인용 Mock 게시글 — 해시태그는 정확히 5개."""
    topic = ", ".join(keywords) if keywords else "오늘의 순간"
    body = (f"{topic} 🌿\n\n"
            "바쁜 일상 속에서 잠시 멈춰 나만의 시간을 즐겨봤어요.\n"
            "작은 여유가 하루를 더 특별하게 만들어 주네요 ✨\n\n"
            "(※ Mock 데이터 — 실제 Gemini 연동 전 예시 문구입니다)")
    tags = []
    for t in [k.replace(" ", "") for k in keywords] + ["일상", "데일리", "소통", "오늘의기록", "인스타그램"]:
        tag = "#" + t.lstrip("#")
        if t and tag not in tags:
            tags.append(tag)
    return {"body": body, "hashtags": tags[:5]}


# ---------- 헤더 ----------
ds("SNS Assistant", "metric")
ds("이미지와 키워드로 SNS 게시글을 자동으로 작성합니다", "subtitle")
st.divider()

# ---------- 입력 영역 ----------
left, right = st.columns(2, gap="large")
with left:
    ds("1. 이미지 업로드", "title")
    files = st.file_uploader("이미지 선택 (여러 장 가능)", type=list(FORMATS),
                             accept_multiple_files=True)
with right:
    ds("2. 키워드 입력", "title")
    raw_kw = st.text_input("키워드 (쉼표로 구분)", placeholder="아메리카노, 감성 카페, 주말 여유")
    keywords = [k.strip() for k in raw_kw.split(",") if k.strip()]
    if keywords:
        ds("입력된 키워드: " + " · ".join(keywords), "label")

    ds("3. 이미지 비율", "title")
    ratio_label = st.radio("SNS 권장 비율", list(RATIOS), label_visibility="collapsed")

# ---------- 크롭 미리보기 + 다운로드 ----------
crops = []  # (파일명, 크롭 이미지, 확장자)
for f in files or []:
    try:
        img = ImageOps.exif_transpose(Image.open(f))   # 휴대폰 사진 회전 정보 반영
    except UnidentifiedImageError:
        st.error(f"❌ 이미지를 열 수 없습니다: {f.name}")
        continue
    ext = Path(f.name).suffix.lower().lstrip(".")
    crops.append((f.name, center_crop(img, RATIOS[ratio_label]), ext))

if crops:
    st.divider()
    ds(f"미리보기 · {ratio_label} · {len(crops)}장", "title")
    slug = ratio_label.split(" ")[0].replace(":", "x") if RATIOS[ratio_label] else "original"
    cols = st.columns(4)
    for i, (name, img, ext) in enumerate(crops):
        with cols[i % 4]:
            st.image(img, width="stretch")
            ds(f"{name} · {img.width}×{img.height}px", "label")
            st.download_button("⬇️ 다운로드", data=encode(img, ext),
                               file_name=f"{Path(name).stem}_{slug}.{ext}",
                               mime=FORMATS[ext][1], key=f"dl-{i}-{name}", width="stretch")

# ---------- 액션 영역 ----------
st.divider()
if st.button("게시글 생성하기", type="primary", disabled=not crops,
             help=None if crops else "이미지를 1장 이상 업로드하세요"):
    st.session_state["post"] = mock_post(keywords)

# ---------- 결과 출력 영역 ----------
post = st.session_state.get("post")
if post and crops:
    ds("생성 결과", "title")
    img_col, card_col = st.columns([1, 2], gap="large")
    with img_col:
        st.image(crops[0][1], width="stretch")
        if len(crops) > 1:
            ds(f"외 {len(crops) - 1}장", "label")
    with card_col:
        with st.container(border=True):
            ds("SNS 게시글", "title")
            ds(post["body"], "text")
            st.markdown(" ".join(f'<span class="ds-bold">{html.escape(t)}</span>'
                                 for t in post["hashtags"]), unsafe_allow_html=True)
            ds(f"해시태그 {len(post['hashtags'])}개 · Mock 데이터", "label")
'''

CONFIG_PATH.parent.mkdir(exist_ok=True)
CONFIG_PATH.write_text(CONFIG_TOML, encoding="utf-8")
APP_PATH.write_text(APP_CODE.lstrip(), encoding="utf-8")
print("\n📝 [2/4] 파일 생성")
print(f"  ✅ {APP_PATH.name}")
print(f"  ✅ {CONFIG_PATH.relative_to(Path.cwd())} (테마: Inter / 라이트·다크 색상)")

# ---------- [4] Streamlit 백그라운드 실행 ----------
print(f"\n🚀 [3/4] Streamlit 백그라운드 실행 (포트 {PORT})")

def stop_streamlit() -> None:
    """Cell 5 가 띄운 Streamlit 서버 종료.
    Windows 의 venv python.exe 는 실제 서버를 자식 프로세스로 띄우므로 프로세스 트리 전체를 종료한다."""
    proc = globals().get("streamlit_proc")
    if proc is None or proc.poll() is not None:
        return
    if sys.platform == "win32":
        subprocess.run(["taskkill", "/PID", str(proc.pid), "/T", "/F"], capture_output=True)
    else:
        proc.terminate()
    proc.wait(timeout=10)
    print("  🔁 이전 Streamlit 서버 종료")

# 셀을 다시 실행하면 이전에 띄운 서버를 먼저 종료 (파일 변경 반영)
stop_streamlit()
time.sleep(1)   # 포트 해제 대기

def lan_ip() -> str | None:
    """이 PC의 내부 네트워크 IP (UDP 소켓으로 라우팅 경로만 조회 — 실제 패킷은 전송하지 않음)."""
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
            s.connect(("8.8.8.8", 80))
            return s.getsockname()[0]
    except OSError:
        return None

def is_up() -> bool:
    try:
        return requests.get(f"{URL}/_stcore/health", timeout=1).ok
    except requests.RequestException:
        return False

server_ok = False
if is_up():
    print(f"  ❌ 포트 {PORT} 를 다른 프로그램(이전 커널의 Streamlit 등)이 사용 중입니다.")
    print("     👉 해당 터미널/프로세스를 종료하거나 커널을 재시작한 뒤 다시 실행하세요.")
else:
    log = LOG_PATH.open("w", encoding="utf-8")
    streamlit_proc = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", str(APP_PATH),
         "--server.port", str(PORT), "--server.address", HOST,
         "--server.headless", "true",
         "--browser.gatherUsageStats", "false"],
        cwd=Path.cwd(), stdout=log, stderr=subprocess.STDOUT,
    )
    for _ in range(30):                       # 최대 30초 대기
        if streamlit_proc.poll() is not None:
            break
        if is_up():
            server_ok = True
            break
        time.sleep(1)

    if server_ok:
        print(f"  ✅ 서버 실행 중 (PID {streamlit_proc.pid}, 로그: {LOG_PATH.name})")
    else:
        print(f"  ❌ 서버 시작 실패 — 로그 마지막 부분:")
        print(textwrap.indent(LOG_PATH.read_text(encoding="utf-8")[-800:], "     "))

# ---------- 접속 안내 ----------
print("\n🌐 [4/4] 접속 안내")
print("=" * 50)
if server_ok:
    print(f"  👉 이 PC에서 접속        :  {URL}")
    ip = lan_ip()
    if ip:
        print(f"  👉 같은 네트워크 기기에서:  http://{ip}:{PORT}")
        print("     (접속이 안 되면 Windows 방화벽에서 Python 의 '개인 네트워크' 인바운드 허용 여부를 확인하세요)")
    print("  · 이미지 여러 장 업로드 → 비율 선택 → 이미지별 다운로드")
    print("  · '게시글 생성하기' → Mock 본문 + 해시태그 5개 카드 확인")
    print("  · 서버 종료: 아래 코드 실행 (커널 재시작만으로는 서버가 남을 수 있음)")
    print("      stop_streamlit()")
    print("=" * 50)
    print("🎉 [성공] UI 프로토타입 실행 완료")
else:
    print("=" * 50)
    print("🚨 [실패] 위 오류 메시지를 확인 후 다시 실행하세요.")